# Greenhouse Gas Emissions Explorer

### 1. Introduction
This dashboard explores trends in greenhouse gas (GHG) emissions by the world's top-20 climate polluting countries using data from the European Commission Joint Research Centre's [Emissions Database for Global Atmospheric Researcher (EDGAR)](https://edgar.jrc.ec.europa.eu/). The dashboard visualizes countries' emissions over time and provides a detailed breakdown of the sectors, sources, and gases that make up those emissions. You can find the finished dashboard at [URL] and the GitHub repo for this project at [URL].

[INSERT SCREENSHOT HERE]

### 2. Library choice

The dashboard was created in Python using [Panel](https://panel.holoviz.org/) and the [HoloViews](https://holoviews.org/index.html) plotting library. Panel is an open-source library for building interactive data dashboards that handles app layout and provides controls for the plots created with HolowViews and the other libraries. I've chosen Panel for this project because it works with a variety of plotting libraries (such as HoloViews and hvPlot) and integrates seamlessly with Jupyter notebooks, making it easy to illustrate the step-by-step process of building the dashboard here in this notebook. The final product in the notebook is served directly, without additional export steps.

Let's start by downloading our libraries and setting our configurations.

In [13]:
import numpy as np
import pandas as pd
import panel as pn
from urllib.request import urlretrieve

import plotly.graph_objs as go
import plotly.io as pio
import matplotlib.pyplot as plt
import matplotlib as mpl
import hvplot.pandas
import warnings
import zipfile
from pathlib import Path

warnings.simplefilter('ignore')
pn.extension('vega', 'plotly', defer_load=True, sizing_mode="stretch_width")
mpl.use('agg')

# Data directory
DATA_DIR_RAW = Path("../data/raw")

### 3. Data
For the dashboard, I'll use the 2026 release of the EDGAR Community GHG emissions database (EDGAR_2026_GHG), available [here](https://edgar.jrc.ec.europa.eu/dataset_ghg2026). The data are provided by the European Union, European Commission, and Joint Research Centre (JRC) under a [Creative Commons Attribution 4.0 International (CC BY 4.0) licence](https://creativecommons.org/licenses/by/4.0/). See the list of sources at the end of this notebook for full citations.

Let's start by downloading EDGAR_2026_GHG from the EC-JRC site and reading it into a Pandas DataFrame. We'll need the annual totals for GHG emissions by sector and country, which are divided into separate files by gas. The gases represent the six main baskets of climate pollutants covered by the [Kyoto Protocol](): carbon dioxide (CO2), methane (CH4), nitrous oxide (N2O), and the so-called F-gases (hydrofluorocarbons, or HFCs, and perfluorocarbons, or PFCs) and sulphur hexafluoride (SF6).  Totals for each gas are weighted by their [global warming potential](https://ec.europa.eu/eurostat/statistics-explained/index.php?title=Glossary:Global-warming_potential_(GWP)) (i.e., their relative potency in trapping heat) and reported in equivalent units of CO2 (CO2e). To get the detailed breakdown of emissions by sector and gas for each of the top-20 countries, we'll need to download and process the data files for each gas.

In [24]:
# Define function for downloading, unzipping, and reading in EDGAR_2026_GHG data
def load_edgar_data(url, sheet="IPCC 2006", data_dir=DATA_DIR_RAW, force_download=False):
    """
    Download the zipped EDGAR data, unzip it, read it, clean it, and output a Pandas DataFrame
    
    Parameters
    url             :   str, the URL containing the EDGAR data
    sheet           :   str, the name of the Excel worksheet with the data we want
    data_dir        :   str, relative path name for the raw data
    force_download  :   bool, force Python to download the zip file even if it already exists

    Returns
    A Pandas DataFrame with the annual emissions data for all countries and sectors
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    dfs = {}
    
    zip_path = data_dir / Path(url).name
    if force_download or not zip_path.exists():
        print(f"Downloading {zip_path.name}...")
        urlretrieve(url, zip_path)
    else:
        print(f"Data file already exists. Using existing file {zip_path.name}")

    with zipfile.ZipFile(zip_path) as zf:
        xlsx_name = next(n for n in zf.namelist() if n.endswith(".xlsx")) # should only have one Excel file
        with zf.open(xlsx_name) as f:
            excel_file = pd.ExcelFile(f)
            df = pd.read_excel(excel_file, sheet_name=sheet, header=9)  # header=9 -> Excel row 10
            df.columns = df.columns.str.lower()
            df = df.drop(columns=["ipcc_annex", "c_group_im24_sh"])
            year_cols = [c for c in df.columns if c.startswith("y_")]
            id_cols = [c for c in df.columns if c not in year_cols]
            df = df.melt(id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="emissions")
            df["year"] = df["year"].str.removeprefix("y_").astype(int)
            df = df.rename(columns={
                "country_code_a3": "country_code",
                "name": "country_name",
                "ipcc_code_2006_for_standard_report": "sector_code",
                "ipcc_code_2006_for_standard_report_name": "sector_name"
            })
            df["units"] = "Gg"
            move_col = df.pop("units")
            df.insert(6, "units", move_col)

    return df

# Get the detailed sectoral data files for each gas
sectors_all_ghg = load_edgar_data(url="https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2026_GHG/EDGAR_AR5_GHG_1970_2025.zip")
sectors_co2 = load_edgar_data(url="https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2026_GHG/IEA_EDGAR_CO2_1970_2025.zip")
sectors_co2_bio = load_edgar_data(url="https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2026_GHG/EDGAR_CO2bio_1970_2025.zip")
sectors_n2o = load_edgar_data(url="https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2026_GHG/EDGAR_N2O_1970_2025.zip")
sectors_fgas = load_edgar_data(url="https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2026_GHG/EDGAR_AR5g_F-gases_1990_2025.zip")

# Look at the CO2 data file
sectors_co2.head()


Data file already exists. Using existing file EDGAR_AR5_GHG_1970_2025.zip
Data file already exists. Using existing file IEA_EDGAR_CO2_1970_2025.zip
Data file already exists. Using existing file EDGAR_CO2bio_1970_2025.zip
Data file already exists. Using existing file EDGAR_N2O_1970_2025.zip
Data file already exists. Using existing file EDGAR_AR5g_F-gases_1990_2025.zip


,country_code,country_name,sector_code,sector_name,substance,fossil_bio,units,year,emissions
0,ABW,Aruba,1.A.1.a,Main Activity Electricity and Heat Production,CO2,fossil,Gg,1970,17.343706
1,ABW,Aruba,1.A.2,Manufacturing Industries and Construction,CO2,fossil,Gg,1970,0.193087
2,ABW,Aruba,1.A.3.a,Civil Aviation,CO2,fossil,Gg,1970,0.754306
3,ABW,Aruba,1.A.3.b,Road Transportation,CO2,fossil,Gg,1970,1.040721
4,ABW,Aruba,1.A.3.d,Water-borne Navigation,CO2,fossil,Gg,1970,NaN


The data files are the granular data for each pollutant/gas and each detailed subsector. We'll also need annual national totals and sector aggregates.

In [25]:
# Function to get annual totals for each gas
def annual_totals(df):
    agg_series = df.groupby(['country_code', 'year'])['emissions'].sum()
    agg_df = pd.DataFrame(agg_series).reset_index()
    agg_df['substance'] = df['substance']
    agg_df['units'] = df['units']
    agg_df = agg_df.reindex(columns=['country_code', 'substance', 'units', 'year', 'emissions'])
    return agg_df

# Function to get annual totals for IPCC aggregate sectors
ipcc_sectors = {

}

total_all_ghg = annual_totals(sectors_all_ghg)
total_co2 = annual_totals(


,country_code,substance,units,year,emissions
0,ABW,GWP_100_AR5_GHG,Gg,1970,38.314617
1,ABW,GWP_100_AR5_GHG,Gg,1971,43.098962
2,ABW,GWP_100_AR5_GHG,Gg,1972,53.909449
3,ABW,GWP_100_AR5_GHG,Gg,1973,58.876864
4,ABW,GWP_100_AR5_GHG,Gg,1974,58.119001


### AI Log

### AI Log

**Entry X:** 
- **Tool**: Claude
- **What I asked for**:
- **What came back**: 
- **What was wrong or could be better**: 
- **How I noticed**:
- **What I did**:

**Entry 1:** Summarize the menu of suggested libraries
- **Tool**: Claude (Sonnet 5.5)
- **What I asked for**: The instructions for this assignment provide a laundry list of Python libraries, some for plotting, some for the widgets/controls to build the dashboard, and some for hosting web apps. I'm familiar with some of these libraries but was unsure which are the most straightforward for integrating into Jupyter. I asked Claude to introduce the libraries, provide URLs for them (including for plotting library galleries), and explain which work well with Jupyter.
- **What came back**: Claude identified Panel/HoloViews, Dash, Gradio, Solara, and Voilà as working in notebooks. It said that Streamlit, Shiny, marimo, and Observable aren't usually run inside Jupyter. It noted that Render offers free hosting for web apps.
- **What was wrong or could be better**: Nothing 
- **How I noticed**: I checked the Panel and HoloView sites and looked through their galleries. Many of the examples are provided as Jupyter notebooks and seemed to be similar to the use case for this assignment. Panel/HoloView also offer flexibility with plotting libraries, so I could create some charts with hvPlot, some with seaborn, and some with Plotly. 
- **What I did**: I decided to go with Panel/Holoview.

**Entry 2:** Downloading data files 
- **Tool**: Claude
- **What I asked for**: Define a function for downloading and extracting zipped Excel files with the EDGAR data. I knew that each of the Excel files was organized the same way. I told Claude exactly what I wanted the function to do (which rows/columns to keep and the steps I wanted for reading it into Pandas, melting it into long format, dropping certain columns, etc).
- **What came back**: 
```
def _read_edgar_sheet(excel_file, sheet_name):
    """Read one EDGAR worksheet (header on Excel row 10) and melt the year columns to long format."""
    df = pd.read_excel(excel_file, sheet_name=sheet_name, header=9)  # header=9 -> Excel row 10
    df.columns = df.columns.str.lower()
    df = df.drop(columns=["ipcc_annex", "c_group_im24_sh"])

    year_cols = [c for c in df.columns if c.startswith("y_")]
    id_cols = [c for c in df.columns if c not in year_cols]
    df = df.melt(id_vars=id_cols, value_vars=year_cols, var_name="year", value_name="emissions")
    df["year"] = df["year"].str.removeprefix("y_").astype(int)
    return df


def load_edgar_data(url=EDGAR_URLs, data_dir=DATA_DIR, force_download=False):
    """Download EDGAR zip files (if needed) and read the sector and country-total sheets.

    Returns a dict of DataFrames named "edgar_sectors_<key>" and "edgar_totals_<key>"
    for each key in `urls`.
    """
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    dfs = {}

    for key, url in urls.items():
        zip_path = data_dir / Path(url).name
        if force_download or not zip_path.exists():
            print(f"Downloading {zip_path.name}...")
            urlretrieve(url, zip_path)
        else:
            print(f"Using existing file {zip_path.name}")

        with zipfile.ZipFile(zip_path) as zf:
            xlsx_name = next(n for n in zf.namelist() if n.endswith(".xlsx"))
            with zf.open(xlsx_name) as f:
                excel_file = pd.ExcelFile(f)
                dfs[f"edgar_sectors_{key}"] = _read_edgar_sheet(excel_file, "IPCC 2006")
                dfs[f"edgar_totals_{key}"] = _read_edgar_sheet(excel_file, "TOTALS BY COUNTRY")

    return dfs
```
- **What was wrong or could be better**: I had originally asked Claude to loop through each of the URLs in a dictionary called EDGAR_URLs, but this made the function too complicated. In the end, I just want it to return a DataFrame
- **How I noticed**: I ran the original function and then tried to call the DataFrames inside the dfs dictionary, but this would just require creating another loop to make each keyed entry into an actual DataFrame.
- **What I did**: I simplified the two functions, combining them into one, and taking out the internal loops. It just returns a DataFrame. I also added some cleaning/simplifying for column names.
